# Qwen + Router Playground (package layout)

_Recovered from a corrupted notebook. The cells below re-create imports, configuration, tuning, benchmarking, and simple visualization utilities._

In [2]:
from routing_fundamentals.qwen_loader import build_embedder, load_generator
from routing_fundamentals.router import load_routes, rank_routes, pick_route
from routing_fundamentals.dispatch import cascade_fallback
from routing_fundamentals.aggregate import semantic_agreement
import yaml

## Load configuration and initialize components

In [ ]:
cfg = yaml.safe_load(open("config.qwen3b.vllm.yaml"))
routes = load_routes("routes.yaml")
embed_cfg = cfg['embedding']
embed_fn = build_embedder(embed_cfg['model'], normalize=embed_cfg.get('normalize', True))
gen = load_generator(cfg['generator'])
W = cfg['router']['weights']
TOP_K = cfg['router']['top_k']
TEMP = cfg['router']['temperature']
print("Routes:", [r.name for r in routes], "backend:", getattr(gen, 'backend', type(gen).__name__))

Routes: ['retrieval.qa', 'code.gen'] backend: vllm-openai


## 1) Interactive Parameter Tuning

Experiment with different routing parameters. Adjust weights and temperature to see how the selected route changes.

In [4]:
def tune_routing_parameters(query, routes, embed_fn, base_weights, base_temp=0.35, variations=None):
    """Test routing with different parameter combinations.

    Returns a dict keyed by variation name.
    """
    if variations is None:
        variations = {
            'temp_low': {'temp': 0.10, 'weights': base_weights},
            'temp_high': {'temp': 0.70, 'weights': base_weights},
            'sim_boost': {'temp': base_temp, 'weights': {**base_weights, 'sim': 2.0}},
            'kw_boost': {'temp': base_temp, 'weights': {**base_weights, 'kw': 1.0}},
        }

    results = {}
    query_vec = embed_fn([query])[0]
    cached = query_vec[None, :]

    for var_name, params in variations.items():
        scored = rank_routes(query, routes, lambda _: cached, params['weights'], top_k=min(TOP_K, len(routes)))
        picked = pick_route(scored, tau=params['temp'])
        results[var_name] = {
            'picked_route': getattr(picked, 'name', str(picked)),
            'scores': [(r.name, c['score']) for r, c in scored],
            'params': params
        }
    return results

# Example usage (uncomment to run)
test_queries = [
    "write a python function to sort a list",
    "explain matrix multiplication",
    "find the documentation for API endpoints",
]

print("=== Parameter Tuning Examples ===")
for query in test_queries:
    print(f"\nQuery: '{query}'")
    results = tune_routing_parameters(query, routes, embed_fn, W, base_temp=TEMP)
    for var_name, result in results.items():
        score_str = [(n, f"{s:.3f}") for n, s in result['scores']]
        print(f"  {var_name}: {result['picked_route']} (scores: {score_str})")


=== Parameter Tuning Examples ===

Query: 'write a python function to sort a list'
  temp_low: code.gen (scores: [('code.gen', '0.432'), ('retrieval.qa', '0.167')])
  temp_high: code.gen (scores: [('code.gen', '0.432'), ('retrieval.qa', '0.167')])
  sim_boost: code.gen (scores: [('code.gen', '0.694'), ('retrieval.qa', '0.183')])
  kw_boost: code.gen (scores: [('code.gen', '0.487'), ('retrieval.qa', '0.167')])

Query: 'explain matrix multiplication'
  temp_low: code.gen (scores: [('code.gen', '0.215'), ('retrieval.qa', '0.209')])
  temp_high: retrieval.qa (scores: [('code.gen', '0.215'), ('retrieval.qa', '0.209')])
  sim_boost: retrieval.qa (scores: [('code.gen', '0.305'), ('retrieval.qa', '0.267')])
  kw_boost: retrieval.qa (scores: [('code.gen', '0.215'), ('retrieval.qa', '0.209')])

Query: 'find the documentation for API endpoints'
  temp_low: retrieval.qa (scores: [('retrieval.qa', '0.642'), ('code.gen', '0.168')])
  temp_high: retrieval.qa (scores: [('retrieval.qa', '0.642'), ('cod

## 2) Performance Analysis

Measure timing and basic complexity of routing operations.

In [5]:
import time

def benchmark_routing(routes, embed_fn, weights, queries, num_runs=5):
    """Benchmark routing performance and return a list of timing dicts."""
    results = []
    for query in queries:
        times = []
        for _ in range(num_runs):
            start = time.time()
            query_vec = embed_fn([query])[0]
            cached = query_vec[None, :]
            scored = rank_routes(query, routes, lambda _: cached, weights, top_k=min(3, len(routes)))
            pick_route(scored, tau=0.35)
            times.append(time.time() - start)
        avg_time = sum(times) / len(times)
        results.append({
            'query': query,
            'avg_time': avg_time,
            'routes_evaluated': len(routes),
            'top_k': min(3, len(routes)),
        })
    return results

# Run performance benchmark (uncomment to re-run)
benchmark_queries = [
    "short query",
    "this is a much longer query about programming and algorithms",
    "write a function",
] * 3  # repeat to stabilize timing

print("\n=== Performance Benchmark ===")
perf_results = benchmark_routing(routes, embed_fn, W, benchmark_queries)
for r in perf_results:
    print(f"{r['avg_time']:.4f}s | top_k={r['top_k']} | routes={r['routes_evaluated']} | '{r['query']}'")
total_time = sum(r['avg_time'] for r in perf_results)
print(f"Total avg time (sum across queries): {total_time:.4f}s")



=== Performance Benchmark ===
0.0136s | top_k=2 | routes=2 | 'short query'
0.0128s | top_k=2 | routes=2 | 'this is a much longer query about programming and algorithms'
0.0126s | top_k=2 | routes=2 | 'write a function'
0.0121s | top_k=2 | routes=2 | 'short query'
0.0126s | top_k=2 | routes=2 | 'this is a much longer query about programming and algorithms'
0.0124s | top_k=2 | routes=2 | 'write a function'
0.0124s | top_k=2 | routes=2 | 'short query'
0.0121s | top_k=2 | routes=2 | 'this is a much longer query about programming and algorithms'
0.0123s | top_k=2 | routes=2 | 'write a function'
Total avg time (sum across queries): 0.1129s


## 3) Basic Visualization

Simple text-based histogram of routing scores for a given query.

In [6]:
def visualize_routing(query, routes, embed_fn, weights, temp=0.35):
    """Print a simple text-based bar chart of route scores for a query."""
    query_vec = embed_fn([query])[0]
    cached = query_vec[None, :]
    scored = rank_routes(query, routes, lambda _: cached, weights, top_k=len(routes))

    print(f"Routing visualization for: '{query}'")
    print("=" * 60)
    max_score = max(c['score'] for _, c in scored)
    min_score = min(c['score'] for _, c in scored)
    score_range = max(max_score - min_score, 1e-9)

    for route, comps in scored:
        normalized = (comps['score'] - min_score) / score_range
        bar_length = int(normalized * 30)
        bar = "█" * bar_length + "░" * (30 - bar_length)
        print(f"{route.name:<24} | score={comps['score']:.3f} | {bar}")

# Example visualization (uncomment to run)
example_query = "write a python function to sort a list"
visualize_routing(example_query, routes, embed_fn, W, temp=TEMP)


Routing visualization for: 'write a python function to sort a list'
code.gen                 | score=0.432 | ██████████████████████████████
retrieval.qa             | score=0.167 | ░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░


## 4) (Optional) Simple Dispatch + Agreement Demo

A tiny sketch showing how one *might* combine dispatch and semantic agreement.

> Note: This is illustrative; adapt to your project's actual generator/route APIs.

In [7]:
def demo_dispatch_and_agreement(query, routes, embed_fn, weights, gen, top_k=3, temp=0.35):
    # Score and pick top-k candidate routes
    scored = rank_routes(query, routes, embed_fn, weights, top_k=min(top_k, len(routes)))
    picked = [r for r, _ in scored]

    # Define simple runner using gen
    def simple_runner(q, gk):
        try:
            text = gen.generate(q, gk)
            return True, text, {}
        except Exception as e:
            return False, str(e), {"error": str(e)}

    runners = {route.name: simple_runner for route in picked}

    gen_kwargs = {
        "max_new_tokens": 512,
        "temperature": temp,
        "top_p": 0.9,
        "top_k": 40,
        "repetition_penalty": 1.05,
    }

    # Dispatch using cascade fallback
    responses, winner = cascade_fallback(query, [r.name for r in picked], runners, gen_kwargs)

    # Compute semantic agreement with proper input format
    successful_responses = {
        name: out for name, ok, out, meta in responses if ok and isinstance(out, str)
    }
    if not successful_responses:
        print("No successful responses")
        return

    agreement_list = semantic_agreement(successful_responses, embed_fn)
    best_key, best_score = max(agreement_list, key=lambda x: x[1])

    print("Picked routes:", [r.name for r in picked])
    print("Winner:", winner)
    for name, ok, out, meta in responses:
        print(f"\nResponse from {name}:")
        if ok:
            print(out)
        else:
            print(f"Failed: {out}")
    print(f"\nSemantic agreement anchor: {best_key}")
    print("Semantic agreement score (max centrality):", best_score)

demo_dispatch_and_agreement("Explain BFS vs DFS and give a Python snippet", routes, embed_fn, W, gen)


ValueError: not enough values to unpack (expected 4, got 2)